# Convolution Transformer: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.conv_transformer_wrapper import ConvTransformerModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "conv_transformer"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El *Convolutional Transformer* se evaluó mediante ventana expansiva con reentrenamiento periódico para pronosticar la volatilidad diaria del crudo Brent.
- En entrenamiento frente a prueba, QLIKE, HMSE y HMAE superaron el umbral de tolerancia por cambios de régimen y pronóstico *one-step forward*, sin evidencia de sobreajuste clásico, con residuos mejor descritos por una distribución $t$ de Student que por la normal.
- En evaluación fuera de muestra, no se encontraron evidencias de autocorrelación lineal ni efectos ARCH en Ljung-Box o ARCH-LM, pero persistió estructura no lineal en la prueba BDS.
- Siguió la dinámica general en periodos de calma con nivel base conservador y subestimó el pico de 2026, resultando más rápido y prudente ante choques extremos que el *Causal Transformer*.
:::

## Introducción

El *Convolution Transformer* (`ConvTransformerModel`) es una arquitectura híbrida de extracción convolucional causal y atención secuencial diseñada para capturar dependencias temporales locales y no lineales a lo largo de una ventana histórica de longitud $L$ (`look_back`), garantizando estrictamente que la predicción en cada instante dependa únicamente de observaciones pasadas y presentes. A diferencia de una autoatención aplicada directamente sobre la proyección punto a punto de cada paso, este enfoque incorpora inicialmente una convolución causal unidimensional (`CausalConv1d`) con relleno exclusivo por la izquierda para sintetizar patrones locales antes de que el *encoder* con máscara causal evalúe las relaciones entre posiciones de la ventana.

En series financieras, esta combinación permite extraer dinámicas locales de corto alcance a través del núcleo convolucional (`conv_kernel_size`) y modelar interacciones no lineales entre pasos temporales mediante autoatención causal, evitando imponer una ecuación paramétrica rígida para la varianza condicional del crudo Brent.

El objetivo de esta sección es presentar, entrenar y evaluar el modelo `ConvTransformerModel` para pronosticar la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el protocolo del *benchmark*. Se analiza el efecto de la longitud de ventana (`look_back`), el tamaño del núcleo convolucional (`conv_kernel_size`), la dimensión del espacio latente (`d_model`), el número de cabezas de atención (`nhead`), la profundidad del *encoder* (`num_layers`), el ancho de la red *feed-forward* (`dim_feedforward`), la función de activación y la regularización (`dropout` y `weight_decay`) sobre el desempeño predictivo, manteniendo las mismas condiciones de información, pérdida QLIKE y evaluación *walk-forward* que el resto de modelos del proyecto.

### Arquitectura

La arquitectura de `ConvTransformerNetwork` se estructura sobre una convolución causal de entrada con relleno izquierdo, activación inicial, codificación posicional aprendible, un *encoder* de Transformer con máscara causal superior (`norm_first=True`), normalización final sobre el último paso temporal y una cabeza de salida positiva.

Dado un tensor de entrada $\mathbf{X}_t \in \mathbb{R}^{L \times F}$ (donde $L$ es `look_back` y $F$ es `in_features`, con $F = 1$ para la serie univariada de retornos estandarizados), la convolución causal unidimensional con relleno izquierdo de tamaño $p = (k - 1) \cdot \text{dilation}$ (con $k$ igual a `conv_kernel_size` y `dilation=1`) y activación $\sigma_{\text{in}}(\cdot)$ (`relu` o `gelu` en `_ACT_MODULES`), sumada a la codificación posicional aprendible $\mathbf{P} \in \mathbb{R}^{1 \times L \times d_{\text{model}}}$, se expresa como:

$$
\mathbf{H}^{(0)} = \sigma_{\text{in}}\left(\text{CausalConv1d}_{k}\left(\mathbf{X}_t\right)\right) + \mathbf{P}.
$$

Para evitar fuga de información hacia adelante (*look-ahead bias*) dentro de la ventana de longitud $L$, se aplica una máscara causal $\mathbf{M} \in \mathbb{R}^{L \times L}$ definida mediante el triángulo superior estricto:

$$
M_{i,j} =
\begin{cases}
0, & \text{si } j \le i, \\
-\infty, & \text{si } j > i.
\end{cases}
$$

En cada capa $l \in \{1, \dots, \text{num\_layers}\}$ del *encoder* con pre-normalización (`norm_first=True`), el mecanismo de atención causal multi-cabeza (*Multi-Head Self-Attention*, MHSA) y el bloque *feed-forward* por posición (FFN) operan residualmente:

$$
\widetilde{\mathbf{H}}^{(l)} = \mathbf{H}^{(l-1)} + \text{MHSA}\left(\text{LayerNorm}\left(\mathbf{H}^{(l-1)}\right), \mathbf{M}\right),
$$

$$
\mathbf{H}^{(l)} = \widetilde{\mathbf{H}}^{(l)} + \text{FFN}\left(\text{LayerNorm}\left(\widetilde{\mathbf{H}}^{(l)}\right)\right),
$$

donde la red *feed-forward* expande la representación hacia `dim_feedforward` (por defecto $2 \cdot d_{\text{model}}$) con *Dropout* y activación seleccionable (`relu` o `gelu` en `_ACTIVATIONS`).

Una vez procesada la secuencia por las capas del *encoder*, se extrae la representación correspondiente al último instante de la ventana, $\mathbf{h}_L^{\text{num\_layers}} \in \mathbb{R}^{d_{\text{model}}}$, se aplica una capa `LayerNorm` final y se proyecta linealmente a un escalar:

$$
\mathbf{z}_L = \text{LayerNorm}\left(\mathbf{h}_L^{\text{num\_layers}}\right), \qquad y_t = \mathbf{w}_{\text{fc}}^{\top}\mathbf{z}_L + b_{\text{fc}}.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva y numéricamente estable, $y_t$ atraviesa la función *Softplus* sumada a un piso $\varepsilon = 10^{-8}$:

$$
\widehat{RS}_t = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ reúne todos los parámetros entrenables de la convolución causal inicial, las incrustaciones posicionales, las capas del *encoder*, las normalizaciones y la capa de salida.

Siguiendo el protocolo del *benchmark*, los retornos logarítmicos diarios $r_t$ se estandarizan mediante transformación Z-score ajustada exclusivamente sobre el segmento de entrenamiento de cada *fold* para evitar fuga de información, mientras que la varianza realizada de Rogers-Satchell se escala mediante el factor de objetivo (`target_scale`). El modelo opera bajo un esquema de predicción de un paso adelante (*one-step-ahead forecasting*) y hereda de `BasePyTorchVolatilityModel` la integración con el *walk-forward* (`retrain_freq=1`), optimización con Adam, *weight decay*, *Dropout* y parada temprana, minimizando la pérdida asimétrica QLIKE:

$$
\boldsymbol{\theta}^{*} = \arg\min_{\boldsymbol{\theta}} \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{QLIKE}}\left(RS_{i}, \widehat{RS}_{i}\right),
$$

donde $N$ es el número de ventanas de entrenamiento y $RS_i$ denota la varianza de Rogers-Satchell objetivo. La implementación se desarrolla en `PyTorch` con diferenciación automática.

## Entrenamiento

Para la selección de hiperparámetros del modelo `ConvTransformerModel`, se mantuvo el mismo protocolo experimental (*Walk-Forward*) y el mismo presupuesto computacional que en los modelos paramétricos, recurrentes y convolucionales. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la configuración convolucional causal y de atención junto con sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema *Walk-Forward* (`n_splits=5`, con `warmup` ajustado según la ejecución y `embargo=1`). Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento y el reentrenamiento (`retrain_freq=1`).

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), la dimensión oculta (`d_model`), el número de cabezas de atención (`nhead`), el número de capas del *encoder* (`num_layers`), el tamaño del núcleo convolucional causal (`conv_kernel_size`), la función de activación (`activation`), la tasa de abandono (`dropout`) y la tasa de aprendizaje (`learning_rate`). Para mantener la consistencia dimensional de la red *feed-forward*, `dim_feedforward` se definió como el doble de `d_model`, y para mitigar el sobreajuste se incorporó también la penalización $\ell_2$ (`weight_decay`).

El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red, bajo un esquema de búsqueda con `n_trials=50` y `early_stop_rounds=15`. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def conv_transformer_search_space(trial: optuna.Trial) -> dict:
    d_model = trial.suggest_categorical("d_model", [32, 64])
    nhead = trial.suggest_categorical("nhead", [2, 4])
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "d_model": d_model,
        "nhead": nhead,
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dim_feedforward": d_model * 2,
        "conv_kernel_size": trial.suggest_categorical("conv_kernel_size", [2, 3, 5]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=ConvTransformerModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=conv_transformer_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

El algoritmo de búsqueda completó 40 ensayos válidos de 41 registrados (97.56 %), alcanzando un mejor valor de QLIKE de `0.5948` con 1 ensayo podado y 0 fallidos. La presencia de 1 ensayo podado refleja un descarte temprano puntual durante la optimización, mientras que el codificador convolucional causal con atención, pre-normalización, conexiones residuales y parada temprana permitió completar el resto de ensayos sin fallos de ejecución.

La búsqueda completa en Optuna tomó aproximadamente 19.25 horas (19 horas y 15 minutos), mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó 665.21 segundos (aproximadamente 11.09 minutos), con un tiempo medio de 3.2610 segundos por partición. Este tiempo de ajuste evidencia el costo computacional de entrenar una arquitectura híbrida convolucional causal y de autoatención con Adam y retropropagación frente a modelos más ligeros, conservando la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [2]:
from pathlib import Path
import optuna
import pandas as pd

resumen_data = []

exec_id = 1
model_tag = "ConvTransformer"
study_name_override = None
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = sorted(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    if "train_time_sec" in df_csv.columns:
        tiempo_total_rep = float(df_csv["train_time_sec"].sum())
        tiempo_promedio_fold = float(df_csv["train_time_sec"].mean())
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        available_studies = optuna.get_all_study_names(storage=storage_url)

        if study_name_override and study_name_override in available_studies:
            target_study = study_name_override
        elif available_studies:
            target_study = available_studies[-1]
        else:
            target_study = f"{model_tag}_study"

        study = optuna.load_study(study_name=target_study, storage=storage_url)

        total_trials = len(study.trials)
        completos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        fallidos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)

        if completos > 0 and study.best_value is not None:
            mejor_qlike = round(float(study.best_value), 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Exec": exec_id,
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Exec,Total Trials,Completos,Podados,Fallidos,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
1,41,40,1,0,0.5948,665.2100,3.2610


El ranking de las cinco mejores configuraciones mostró un consenso en el uso de ventanas de 21 días (`look_back = 21`) en cuatro de los cinco casos, un espacio latente compacto (`d_model = 32`, `dim_feedforward = 64`), 4 cabezas de atención (`Heads = 4`) y un núcleo convolucional causal de tamaño 3 (`Kernel = 3`), lo que sugiere que el historial de tres semanas con extracción local de tamaño 3 resulta informativo para capturar dinámicas recientes en la volatilidad del Brent. En cuanto a la profundidad del *encoder*, predominó una capa (`Capas = 1`) frente a dos capas en el segundo puesto, mientras que las activaciones se repartieron entre `relu` y `gelu`. Al carecer de un estado oculto recurrente como las redes secuenciales tradicionales, la combinación de convolución causal local y atención causal sobre la ventana de entrada permite ponderar relaciones entre pasos temporales dentro del contexto seleccionado sin recurrencia paso a paso.

In [6]:
csv_path = exec_dir / "ConvTransformer_results.csv"
if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("*_results.csv"))
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_ct = pd.read_csv(csv_path)

df_ct["lr"] = df_ct["learning_rate"].apply(lambda x: f"{x:.4f}")
df_ct["wd"] = df_ct["weight_decay"].apply(lambda x: f"{x:.1e}")
df_ct["drop_fmt"] = df_ct["dropout"].apply(lambda x: f"{x:.3f}")

cols_agrupar = [
    "look_back", "d_model", "nhead", "num_layers",
    "dim_feedforward", "conv_kernel_size", "drop_fmt",
    "activation", "lr", "wd"
]

df_agg = (
    df_ct.groupby(cols_agrupar, as_index=False)["qlike"]
    .agg(mean="mean", std="std")
)

df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

df_top5 = df_agg.sort_values(by="mean", ascending=True).head(5).rename(
    columns={
        "look_back": "Lookback",
        "d_model": "d_model",
        "nhead": "Heads",
        "num_layers": "Capas",
        "dim_feedforward": "FFN",
        "conv_kernel_size": "Kernel",
        "drop_fmt": "Dropout",
        "activation": "Act",
        "lr": "LR",
        "wd": "Weight Decay",
    }
)

cols_finales = [
    "Lookback", "d_model", "Heads",
    "Capas", "FFN", "Kernel",
    "Dropout", "Act", "LR", "Weight Decay", "QLIKE"
]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,d_model,Heads,Capas,FFN,Kernel,Dropout,Act,LR,Weight Decay,QLIKE
21,32,4,1,64,3,0.208,relu,0.0003,9.7e-04,0.595 ± 0.186
21,32,4,2,64,3,0.229,gelu,0.0005,4.1e-04,0.597 ± 0.123
21,32,4,1,64,3,0.298,relu,0.0003,9.0e-04,0.608 ± 0.164
14,32,4,1,64,3,0.228,gelu,0.0007,9.9e-04,0.611 ± 0.115
21,32,4,1,64,3,0.216,gelu,0.0004,8.2e-04,0.614 ± 0.119


Entre las dos primeras configuraciones, la diferencia de QLIKE es muy reducida frente a la dispersión observada entre *folds*, por lo que ambas resultan cercanas en validación. La elección de la primera configuración responde a su menor pérdida media de validación (`0.595`) y a un criterio de parsimonia al emplear una sola capa (`Capas = 1`) en lugar de dos, reduciendo la complejidad y la carga paramétrica del modelo sin sacrificar capacidad predictiva.

In [ ]:
best_conv_transformer_model = ConvTransformerModel(
    look_back=21,
    d_model=32,
    nhead=4,
    num_layers=1,
    dim_feedforward=64,
    conv_kernel_size=3,
    dropout=0.208,
    activation="relu",
    learning_rate=0.0003,
    weight_decay=9.7e-4,
    max_epochs=200,
    patience=20,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_conv_transformer_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time

El ajuste del modelo sobre toda la muestra de entrenamiento tuvo una duración de 12.56 segundos (12.5600 s). En la gráfica de aprendizaje, ambas trayectorias descendieron de forma suave y progresiva desde el inicio sin picos iniciales abruptos, alcanzando el mínimo de validación en la época 21, donde se seleccionaron los mejores pesos, y continuando hasta la época 41 tras activarse la parada temprana.

Durante las primeras veinte épocas, la pérdida de validación se mantuvo por debajo de la de entrenamiento, y solo después de la época 21 (alrededor de las épocas 25–26) ambas curvas se cruzaron y estabilizaron en mesetas cercanas. Este comportamiento sugirió una convergencia ordenada del bloque convolucional causal y de autoatención, con una ligera brecha posterior a favor del ajuste en entrenamiento pero sin divergencia en validación, lo que indicó una captura estable del nivel base de la varianza condicional con margen acotado sobre eventos extremos.

In [11]:
import pandas as pd

history_csv_path = exec_dir / "conv_transformer_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

n_epochs = len(history_df)
x_max = max(n_epochs + 1, 2)
x_step = max(1, round(x_max / 11))

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14, line_width=2.0,
    title_size=14, label_size=14, tick_size=12,
    legend_size=14, ylim=(0, 6), xlim=(0, x_max),
    y_step=0.5, x_step=x_step, legend_bbox=(0.95, 0.95)
)

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no fue posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) sobre la serie completa de entrenamiento: primero se transformó `y_train` con el `StandardScaler` ajustado durante `fit` y se escaló el objetivo de varianza, luego se construyó la matriz de rezagos con `TimeSeriesDataset` usando ventanas de 21 días (`look_back = 21`) y finalmente se evaluó en modo `eval()` sin gradientes ni regularización. Las varianzas estimadas se reescalaron a su unidad original y se alinearon con los datos reales descartando las observaciones iniciales correspondientes a la ventana.

In [ ]:
best_conv_transformer_model.net.eval()
look_back = best_conv_transformer_model.context_length
target_scale = getattr(best_conv_transformer_model, "target_scale", 1.0)

y_train_scaled = best_conv_transformer_model.scaler.transform(
    np.asarray(y_train).reshape(-1, 1)
).flatten()
rs_train_scaled = np.asarray(rs_train) * target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=getattr(best_conv_transformer_model, "batch_size", 128),
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_conv_transformer_model.device)
        preds_batch = (
            best_conv_transformer_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        )
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / target_scale
n_preds = len(preds_train_var)

start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
y_train_aligned = np.asarray(y_train)[start_idx : start_idx + n_preds]
rs_train_aligned = np.asarray(rs_train)[start_idx : start_idx + n_preds]

df_train_preds = pd.DataFrame(
    {
        "y_true": y_train_aligned,
        "rs_true": rs_train_aligned,
        "y_pred": preds_train_var,
    },
    index=dates_aligned,
)

train_output_path = exec_dir / "conv_transformer_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=True)

A partir de las varianzas condicionales predichas $\left(\hat{\sigma}_t^2\right)$ y de los retornos observados $\left(y_t\right)$, se evaluó el ajuste del modelo sobre el conjunto de entrenamiento mediante los residuos estandarizados:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El diagnóstico sobre los residuos estandarizados mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en el rezago 10 ($p_{\text{adj}} = .754$), pero detectó autocorrelación significativa en los rezagos 20 y 30 ($p_{\text{adj}} = .004$ en ambos casos). Las pruebas ARCH-LM rechazaron la hipótesis nula de homocedasticidad condicional en los rezagos 10, 20 y 30 ($p_{\text{adj}} < .001$), evidenciando persistencia de agrupamiento de volatilidad dentro de la muestra de entrenamiento, y la prueba BDS en dimensión 2 también indicó estructura no lineal significativa ($p < .001$). Asimismo, las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$), reflejando colas pesadas características de los retornos del crudo Brent.

In [3]:
import numpy as np
import pandas as pd

train_output_path = exec_dir / "conv_transformer_train_predictions.csv"

df_train_preds = pd.read_csv(train_output_path)
rs_train_aligned = df_train_preds["rs_true"].to_numpy()
preds_train_var = df_train_preds["y_pred"].to_numpy()

n_preds = len(preds_train_var)
y_arr = np.asarray(y_train).reshape(-1)
y_train_aligned = y_arr[-n_preds:]

std_residuals_train = y_train_aligned / np.sqrt(np.maximum(preds_train_var, 1e-12))
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),12.5285,.251,.754,Sin Autocorrelación
Ljung-Box (Lag 20),44.0455,.001,.004,Autocorrelación Significativa
Ljung-Box (Lag 30),58.3162,.001,.004,Autocorrelación Significativa
ARCH-LM (Lag 10),140.9338,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),178.1075,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),254.7730,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,3514.4294,< .001,--,Residuos No Normales
Lilliefors,0.0779,< .001,--,Residuos No Normales
BDS (Dim 2),6.4252,< .001,--,Estructura No Lineal Significativa


La persistencia de efectos ARCH residuales en los rezagos evaluados indica que el *Convolution Transformer* tuvo una capacidad limitada para absorber por completo el agrupamiento de volatilidad, en mayor medida que arquitecturas como la TCN o el *Causal Transformer*. Dado que el objetivo del modelo es capturar la dinámica temporal de la volatilidad y estandarizar adecuadamente los choques, esta heterocedasticidad residual señala una reacción parcial o rezagada ante cambios de régimen y picos de volatilidad.

La inspección visual de los residuos estandarizados muestra un ajuste general adecuado a la distribución $t$ de Student con $\nu = 3.3515$ grados de libertad. Los cuantiles muestrales siguen de cerca la diagonal teórica en el centro y buena parte de la cola inferior, observándose desviaciones moderadas por debajo de la línea de referencia en los cuantiles superiores más altos. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .5206$; Cramér-von Mises: $p = .3435$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [13]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.8),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal ($p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 383.01$ y $\Delta\text{BIC} = 377.33$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados del modelo en la muestra de entrenamiento en comparación con el ajuste Gaussiano.

In [14]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-2,953.32","5,910.65","5,921.99",-,-
t-Student,"-2,760.82","5,527.64","5,544.66",385.00,< .001


A diferencia del *Causal Transformer*, que reaccionaba con rapidez e incluso llegaba a sobreestimar picos en episodios de alta volatilidad, el *Convolutional Transformer* muestra un comportamiento más conservador: en periodos normales su predicción tiende a colapsar hacia un nivel base cercano a la media, un rasgo que en este estudio solo se había observado en arquitecturas recurrentes, mientras que en choques severos como la crisis de 2020 subestima la magnitud de los picos de volatilidad realizada.

Este comportamiento puede deberse a que la convolución causal inicial (`CausalConv1d` con núcleo de tamaño 3) suaviza y promedia las perturbaciones locales antes de que entren a la autoatención, y a que la combinación de un espacio latente compacto (`d_model = 32`), una sola capa de *encoder* y la pérdida asimétrica QLIKE favorece estimaciones prudentes que evitan penalizaciones severas por sobrepredicción, a costa de atenuar los saltos extremos.

In [17]:
look_back = 21
n_preds = len(preds_train_var)
start_idx = len(y_train) - n_preds

dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(
    np.maximum(np.asarray(rs_train)[start_idx : start_idx + n_preds], 0.0)
)
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Conv Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Conv Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su ajuste sobre los datos observados con su capacidad predictiva en observaciones no vistas. Mientras que el conjunto de entrenamiento permite verificar la convergencia del algoritmo sobre la dinámica histórica, la evaluación fuera de muestra determina el desempeño del modelo ante nuevos regímenes de mercado.

Para diagnosticar posibles problemas de generalización, se aplicaron los criterios pre-registrados del protocolo experimental a partir de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{\text{abs}} = M_{\text{test}} - M_{\text{train}}
$$

Se fijó un umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ para las métricas evaluadas, de modo que discrepancias por encima de este margen señalan cambios relevantes entre el ajuste en muestra y el desempeño fuera de muestra.

En el modelado de volatilidad con pérdida QLIKE, esta lectura debe matizarse por la asimetría de la función de pérdida, que castiga con mayor severidad la subestimación de la varianza que su sobreestimación. Un modelo con sesgo alcista o defensivo puede registrar un QLIKE menor en prueba que en entrenamiento sin que ello implique una mejor capacidad estructural, sino una coincidencia favorable entre el nivel predicho y el régimen de volatilidad del periodo de prueba. Por ello, el análisis de $\Delta_{\text{abs}}$ se complementa con la inspección de las trayectorias predichas y el diagnóstico de residuos estandarizados.

Las predicciones fuera de muestra se generaron mediante el método `forecast` del modelo, aplicando una ventana expansiva con reentrenamiento periódico sin fuga de información y devolviendo las varianzas condicionales en su escala original para compararlas con la volatilidad realizada.

In [ ]:
start_time = time.time()
preds_test = best_conv_transformer_model.forecast(y_seq=y_test, rs_seq=rs_test)
preds_time = time.time() - start_time

print(f"Tiempo de pronóstico: {preds_time / 60:.2f} minutos ({preds_time:.2f} s)")

preds_test_arr = np.asarray(preds_test).flatten()
rs_test_arr = np.asarray(rs_test).flatten()
y_test_arr = np.asarray(y_test).flatten()

n_test = len(preds_test_arr)
dates_test_aligned = df_test.index[-n_test:]
rs_test_aligned = rs_test_arr[-n_test:]
y_test_aligned = y_test_arr[-n_test:]

df_preds = pd.DataFrame(
    {
        "date": dates_test_aligned,
        "y_true": y_test_aligned,
        "rs_true": rs_test_aligned,
        "y_pred": preds_test_arr,
    }
)

output_path = exec_dir / "conv_transformer_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 62.80 minutos (3768.00 s) para las 953 observaciones del conjunto de prueba, lo que equivale a un tiempo promedio de 0.07 minutos (unos 3.95 segundos) por observación. Este tiempo es sustancialmente superior al de los modelos paramétricos, cuyo cómputo toma milisegundos, reflejando el costo computacional de actualizar y reajustar los pesos del *Conv Transformer* a lo largo de la ventana de evaluación.

El diagnóstico de los residuos estandarizados fuera de muestra no mostró indicios de autocorrelación tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} = 1.000$, $.712$ y $1.000$, respectivamente), así como ausencia de efectos ARCH significativos en los tres rezagos evaluados ($p_{\text{adj}} = 1.000$ en los rezagos 10, 20 y 30). En cambio, la prueba BDS en dimensión 2 detectó estructura no lineal significativa ($p = .016$), y las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$ en ambos casos).

In [4]:
preds_path = exec_dir / "conv_transformer_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values

std_residuals_oos = y_test / np.sqrt(np.maximum(preds_test_var, 1e-12))
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),10.8106,.372,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),24.1182,.237,.712,Sin Autocorrelación
Ljung-Box (Lag 30),28.0851,.566,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),11.1820,.344,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),18.6715,.543,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),24.3059,.758,1.000,Sin Efectos ARCH
Jarque-Bera,360.2594,< .001,--,Residuos No Normales
Lilliefors,0.0491,< .001,--,Residuos No Normales
BDS (Dim 2),-2.3981,.016,--,Estructura No Lineal Significativa


La inspección visual de los residuos estandarizados fuera de muestra muestra un ajuste general adecuado a la distribución $t$ de Student con $\nu = 5.2574$ grados de libertad. Los cuantiles muestrales siguen de cerca la diagonal teórica a lo largo del centro y gran parte de los extremos, observándose únicamente leves desviaciones por encima de la línea de referencia en los cuantiles superiores más altos. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .5802$; Cramér-von Mises: $p = .5539$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [5]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal ($p < .001$). Los criterios de información mostraron una reducción de $\Delta\text{AIC} = 78.24$ y $\Delta\text{BIC} = 73.38$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados del modelo fuera de muestra en comparación con el ajuste Gaussiano.

In [6]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,527.52","3,059.04","3,068.76",-,-
t-Student,"-1,487.40","2,980.80","2,995.38",80.24,< .001


En la comparación entre entrenamiento y prueba, el modelo excedió el umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ en QLIKE ($\Delta_{\text{abs}} = -0.1238$), en HMSE ($\Delta_{\text{abs}} = +3.2683$) y en HMAE ($\Delta_{\text{abs}} = +0.1125$), mientras que el resto de las métricas (CRPS, pérdidas *pinball* en $\tau = 0.05$ y $\tau = 0.95$, MSE, RMSE y MAE) permanecieron dentro del margen de tolerancia y en magnitudes bajas. Que estas diferencias superen el umbral no evidencia un sobreajuste clásico: este orden en las métricas responde a cambios en el régimen de volatilidad entre muestras y al pase a pronóstico secuencial *one-step forward* fuera de muestra. La caída pronunciada de QLIKE en prueba (de `0.6031` a `0.4793`), un patrón ya observado anteriormente pero en esta ocasión de mayor magnitud, se asocia a la transición hacia la predicción secuencial *one-step forward*, a la asimetría de la función de pérdida y al régimen de volatilidad del periodo de evaluación, mientras que los incrementos en HMSE y HMAE reflejan el mayor peso relativo de las discrepancias en regímenes de baja varianza o choques puntuales.

In [9]:
look_back = 21
n_preds = len(preds_train_var)
rs_train_aligned = np.asarray(rs_train)[-n_preds:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia",
    ),
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.6031,0.4793,-0.1238
CRPS,0.0131,0.0124,-0.0006
Pinball (τ=0.05),0.0031,0.0026,-0.0005
Pinball (τ=0.95),0.0018,0.0016,-0.0001
MSE,0.0000,0.0000,+0.0000
RMSE,0.0019,0.0022,+0.0003
MAE,0.0006,0.0005,-0.0001
HMSE,1.9602,5.2286,+3.2683
HMAE,0.7331,0.8457,+0.1125


La inspección visual de los pronósticos fuera de muestra muestra que el *Conv Transformer* sigue en términos generales la tendencia y el nivel base de la volatilidad realizada durante los periodos de relativa calma entre 2023 y principios de 2026, pero presenta limitaciones marcadas ante choques extremos, subestimando el pico principal observado en 2026 (donde $\sigma_R$ alcanza cerca de `0.25` mientras $\hat{\sigma}_t$ apenas responde con un salto menor y luego presenta oscilaciones rezagadas o por encima en episodios posteriores dentro del acercamiento). Este comportamiento es coherente con los resultados numéricos previos: la ausencia de efectos ARCH residuales en los rezagos evaluados indica que el modelo absorbió la mayor parte de la autocorrelación condicional ordinaria, pero el incremento de HMSE (`5.2286`) y HMAE (`0.8457`) en prueba refleja las discrepancias relativas frente a valores pequeños y la incapacidad de ajustar con precisión la magnitud instantánea de los saltos abruptos.

In [10]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(np.maximum(df_preds_test["rs_true"], 0.0))
pred_test_vol = np.sqrt(np.maximum(df_preds_test["y_pred"], 0.0))

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (
    df_preds_test["date"] <= "2026-06-30"
)

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste Fuera de la Muestra (Conv Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Conv Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La implementación y evaluación del *Convolutional Transformer* permitieron contrastar el desempeño de una arquitectura híbrida de convolución causal y autoatención para el pronóstico de la volatilidad diaria del crudo Brent. En la comparación entre entrenamiento y prueba, el modelo superó el umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ en QLIKE ($\Delta_{\text{abs}} = -0.1238$), HMSE ($\Delta_{\text{abs}} = +3.2683$) y HMAE ($\Delta_{\text{abs}} = +0.1125$), sin que ello evidencie un sobreajuste clásico, dado que estas diferencias responden a cambios de régimen de volatilidad y al paso a pronóstico secuencial *one-step forward*, mientras que el resto de las métricas se mantuvieron dentro del margen y en rangos bajos.

En el diagnóstico fuera de muestra, los residuos estandarizados no mostraron autocorrelación lineal significativa en Ljung-Box ni efectos ARCH significativos en los rezagos 10, 20 y 30 tras el ajuste de Bonferroni, aunque la prueba BDS en dimensión 2 sí detectó estructura no lineal significativa ($p = .016$). Asimismo, se rechazó la normalidad en Jarque-Bera y Lilliefors, y los residuos presentaron colas pesadas mejor representadas por una distribución $t$ de Student ($\nu = 5.2574$) que por la normal, con pruebas de Kolmogorov-Smirnov y Cramér-von Mises favorables a este ajuste.

En términos de trayectoria y cómputo, el modelo siguió la tendencia general en periodos de calma pero tendió a un nivel base conservador y subestimó el pico de volatilidad de 2026, mientras que el pronóstico secuencial con reentrenamiento periódico tomó 62.80 minutos (3768.00 s) para las 953 observaciones de prueba (0.07 minutos o 3.95 segundos por observación en promedio). En conjunto, el *Convolutional Transformer* ofrece un ajuste estable del nivel base de volatilidad y menor tiempo de cómputo fuera de muestra que el *Causal Transformer*, pero presenta menor reactividad ante choques extremos que arquitecturas como la TCN o el *Causal Transformer*.